# Remaining eight Rodinia applications: GPU capture
Select **Runtime → Change runtime type → T4 GPU**, then **Run all**. Download the final ZIP even if some cases fail.

This is a first GPU compatibility/collection pass, not a claim that all eight already work. It retains original application drivers. The analyzer DAGs for these eight are still pending; these inputs are recorded for matching them later.

Collects one Nsight Systems trace and a separate Nsight Compute capture per successful application. Hardware metrics are `sm__cycles_elapsed.avg` and `sm__cycles_active.avg`, **per kernel, averaged across SMs—not whole-application cycles**. Transfer timestamps remain in Systems traces. No time-to-cycle conversion. Counter permission errors are retained; they cannot be repaired by opening the report on a laptop.

Legacy texture APIs require CUDA 11.8. Installation may take several minutes. Only a toolkit is installed, not a GPU driver. Data for the video and myocyte applications comes from the original Rodinia archive; its download may be large. Build/run failures are isolated and logged. Numerical correctness and source/GPU DAG equivalence require subsequent validation.


In [ ]:
from pathlib import Path
import datetime, subprocess, shutil, json, os, sys, re, hashlib, tarfile, sqlite3, csv
RUN = Path('/content/rodinia_remaining') / datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN.mkdir(parents=True, exist_ok=True)
LOGS = RUN/'logs'; LOGS.mkdir(exist_ok=True)
def run(cmd, label, cwd=None, check=True, timeout=900):
    cmd=list(map(str,cmd))
    (LOGS/(label+'.command.json')).write_text(json.dumps(cmd))
    try:
        p=subprocess.run(cmd,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=timeout)
        rc,text=p.returncode,p.stdout
    except (OSError,subprocess.TimeoutExpired) as e: rc,text=-1,str(e)
    (LOGS/(label+'.log')).write_text(text)
    print(label,rc,text[-1200:])
    if check and rc: raise RuntimeError(label+' failed; see logs and download the ZIP.')
    return rc,text
assert shutil.which('nvidia-smi'), 'Select a GPU runtime first.'
run(['nvidia-smi'],'gpu')
_,cap=run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],'cap')
ARCH=cap.splitlines()[0].strip().replace('.','')
assert ARCH=='75', 'This notebook targets T4 (sm_75). Select T4 to preserve legacy CUDA compatibility.'
REPO=RUN/'repo'
run(['git','clone','--depth','1','--branch','pa2746-byte-sampleworkloads','https://github.com/pa2746-byte/ESP_workload.git',REPO],'clone')
run(['git','-C',REPO,'rev-parse','HEAD'],'source_commit')
sys.path.insert(0,str(REPO/'tools'))
from capture_rodinia_cycles import locate, parse_cycles, METRICS


In [ ]:
# Use the NVIDIA repository already configured in Colab.
run(['apt-get','update','-qq'],'apt_update',check=False)
run(['apt-get','install','-y','cuda-toolkit-11-8','gcc-11','g++-11','make','libgl1-mesa-dev','libglu1-mesa-dev'],'toolchain_install',check=False,timeout=1800)
CUDA=Path('/usr/local/cuda-11.8')
assert (CUDA/'bin/nvcc').exists(), 'CUDA 11.8 installation failed. Download logs; do not build legacy textures with CUDA 12+.'
os.environ['PATH']=str(CUDA/'bin')+':'+os.environ['PATH']
os.environ['LD_LIBRARY_PATH']=str(CUDA/'lib64')+':'+os.environ.get('LD_LIBRARY_PATH','')
for tool,prefix in [('nsys','nsight-systems'),('ncu','nsight-compute')]:
    if locate(tool): continue
    _,listing=run(['apt-cache','search','--names-only','^'+prefix],tool+'_packages',check=False)
    names=[l.split(' - ')[0].strip() for l in listing.splitlines()]
    names=[n for n in names if re.fullmatch(re.escape(prefix)+r'(?:-cli)?(?:-\d[\d.]*)?',n)]
    if names:
        package=max(names,key=lambda n:tuple(map(int,re.findall(r'\d+',n))) or (0,))
        run(['apt-get','install','-y',package],tool+'_install',check=False)
NSYS,NCU=locate('nsys'),locate('ncu')
run([CUDA/'bin/nvcc','--version'],'nvcc')
for name,path in [('nsys',NSYS),('ncu',NCU)]:
    if path: run([path,'--version'],name+'_version',check=False)
print('Collectors:',NSYS,NCU)


In [ ]:
# Stage original sources; keep all edits and inputs in the evidence bundle.
BASE=RUN/'staged'; SRC=BASE/'cuda'; DATA=BASE/'data'
SRC.mkdir(parents=True,exist_ok=True); DATA.mkdir(exist_ok=True)
CASES=['dwt2d','heartwall','huffman','hybridsort','leukocyte','mummergpu','myocyte','streamcluster']
for case in CASES:
    shutil.copytree(REPO/'workloads/rodinia'/case,SRC/case,dirs_exist_ok=True)
# Original video and numerical input data. Extract regular files only, beneath data/.
URL='https://www.cs.virginia.edu/~skadron/lava/rodinia/Packages/rodinia_3.1.tar.bz2'
archive=Path('/content/rodinia_3.1.tar.bz2')
rc,_=run(['curl','-fL','--retry','2','--max-time','900',URL,'-o',archive],'dataset_download',check=False,timeout=1000)
if rc==0:
    try:
        with tarfile.open(archive) as tf:
            for member in tf:
                parts=Path(member.name).parts
                if not member.isfile() or 'data' not in parts: continue
                tail=parts[parts.index('data')+1:]
                if not tail or tail[0] not in {'heartwall','leukocyte','myocyte'} or '..' in tail: continue
                target=DATA.joinpath(*tail); target.parent.mkdir(parents=True,exist_ok=True)
                with tf.extractfile(member) as inp, target.open('wb') as out: shutil.copyfileobj(inp,out)
    except Exception as e: print('Dataset extraction failed:',e)
# Deterministic nonempty synthetic DWT input; other cases use their application generators or upstream data.
(DATA/'dwt2d').mkdir(exist_ok=True)
(DATA/'dwt2d/input.rgb').write_bytes(bytes(range(256))*48) # 64*64*3
# Compatibility changes apply only to staged source copies.
changes=[]
for case in CASES:
    for p in (SRC/case).rglob('*'):
        if p.suffix not in {'.cu','.c','.h','.cpp','.cuh'}: continue
        text=p.read_text(errors='replace'); new=text.replace('cudaThreadSynchronize','cudaDeviceSynchronize').replace('cudaThreadExit','cudaDeviceReset')
        if new!=text:
            p.write_text(new); changes.append({'file':str(p.relative_to(BASE)),'change':'Rename deprecated synchronization/reset API'})
(BASE/'compatibility_changes.json').write_text(json.dumps(changes,indent=2))
(BASE/'common').mkdir(exist_ok=True)
(BASE/'common/make.config').write_text(f'CUDA_DIR = {CUDA}\nCUDA_LIB_DIR = {CUDA}/lib64\nSDK_DIR = {CUDA}\n')
compat=BASE/'compat.h'
compat.write_text('#include <cuda_runtime.h>\n#include <cstring>\n#include <cstdlib>\n#include <cmath>\n#include <ctime>\n')


In [ ]:
# Explicit application build recipes; no replacement kernels.
NV=[str(CUDA/'bin/nvcc'),'-ccbin','g++-11','-O3','-lineinfo','-std=c++14','-arch=sm_75','-include',str(compat)]
plans={
 'dwt2d':(['main.cu','dwt.cu','components.cu','dwt_cuda/fdwt53.cu','dwt_cuda/fdwt97.cu','dwt_cuda/common.cu','dwt_cuda/rdwt97.cu','dwt_cuda/rdwt53.cu'],['-d','64x64','-c','3','-l','1','-5','input.rgb']),
 'huffman':(['main_test_cu.cu','stats_logger.cpp','cpuencode.cpp'],[]),
 'hybridsort':(['bucketsort.cu','mergesort.cu','main.cu'],['r']),
 'myocyte':(['main.cu'],['1','1','0']),
 'streamcluster':(['streamcluster_cuda_cpu.cpp','streamcluster_cuda.cu'],['2','4','8','128','128','32','none','output.txt','1']),
 'mummergpu':(['src/mummergpu.cu','src/mummergpu_gold.cpp','src/suffix-tree.cpp','src/PoolMalloc.cpp','src/mummergpu_main.cpp'],['data/shortref.fa','data/shortqry.fa']),
}
status={}; cycles=[]
for case in CASES:
    s=status[case]={'trace_collected':False,'cycles_measured':False,'analyzer_comparison':'pending'}
    wd=SRC/case; exe=wd/'capture_app'
    try:
        if case in {'heartwall','leukocyte','myocyte'}:
            required={'heartwall':['test.avi'],'leukocyte':['testfile.avi'],'myocyte':['y.txt','params.txt']}[case]
            for f in required:
                assert (DATA/case/f).is_file(), f'Missing original input: {case}/{f}; dataset download/extraction must succeed.'
        if case=='heartwall':
            for f in ['avilib','avimod']:
                run(['gcc-11','-O3','-c',f'AVI/{f}.c','-o',f'AVI/{f}.o'],case+'_'+f,wd)
            run([*NV,'-IAVI','main.cu','AVI/avilib.o','AVI/avimod.o','-o',exe],case+'_build',wd)
            args=['../../data/heartwall/test.avi','2']
        elif case=='leukocyte':
            run(['make','NVCC='+str(CUDA/'bin/nvcc')+' -ccbin g++-11 -arch=sm_75 -lineinfo','CC=gcc-11'],case+'_build',wd)
            exe=wd/'CUDA/leukocyte'; args=['../../data/leukocyte/testfile.avi','2']
        else:
            files,args=plans[case]
            extra=['-Xcompiler','-fopenmp','-lcuda'] if case=='mummergpu' else []
            run([*NV,*files,*extra,'-o',exe],case+'_build',wd)
        s['built']=True; s['arguments']=args; s['cwd']=str(wd)
        command=[str(exe),*args]
        if NSYS:
            prefix=RUN/case
            rc,_=run([NSYS,'profile','--trace=cuda','--sample=none','--cpuctxsw=none','--force-overwrite=true','-o',prefix,*command],case+'_nsys',wd,False)
            s['nsys_returncode']=rc
            report=prefix.with_suffix('.nsys-rep')
            if report.exists():
                db=prefix.with_suffix('.sqlite')
                erc,_=run([NSYS,'export','--type=sqlite','--force-overwrite=true','--output',db,report],case+'_export',check=False)
                if erc==0 and db.exists():
                    with sqlite3.connect(db) as conn:
                        tables={r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
                        count=conn.execute('SELECT COUNT(*) FROM CUPTI_ACTIVITY_KIND_KERNEL').fetchone()[0] if 'CUPTI_ACTIVITY_KIND_KERNEL' in tables else 0
                    s['kernel_count']=count; s['trace_collected']=rc==0 and count>0
        else: s['trace_error']='Nsight Systems unavailable'
        if NCU:
            path=RUN/(case+'_ncu.csv')
            rc,_=run([NCU,'--metrics',','.join(METRICS),'--clock-control','none','--cache-control','none','--replay-mode','kernel','--csv','--page','details','--print-metric-name','name','--print-units','base','--log-file',path,'--export',RUN/(case+'_cycles'),'--force-overwrite',*command],case+'_ncu',wd,False)
            rows=parse_cycles(path,case) if path.exists() else []
            s['ncu_returncode']=rc; s['cycles_measured']=rc==0 and bool(rows)
            if s['cycles_measured']: cycles.extend(rows)
        else: s['cycle_error']='Nsight Compute unavailable'
    except Exception as e: s['error']=str(e)
    finally:
        (RUN/'status.json').write_text(json.dumps(status,indent=2))
        print(case,json.dumps(s))
with (RUN/'cycles.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=['workload','launch_id','kernel','metric','unit','value']); writer.writeheader(); writer.writerows(cycles)
# Hash exact sources and inputs for matching source analysis later.
hashes={str(p.relative_to(BASE)):hashlib.sha256(p.read_bytes()).hexdigest() for p in BASE.rglob('*') if p.is_file()}
(RUN/'staged_sha256.json').write_text(json.dumps(hashes,indent=2))
print(json.dumps(status,indent=2))


## Download (run even after an error)
Send the ZIP back. It contains reports where successful, exact staged sources and inputs, cycle CSVs, commands, and errors. A trace with kernels is collection evidence, not proof of numerical correctness or matching dependency DAGs. The eight analyzer graphs are not included because they are not validated yet.


In [ ]:
from google.colab import files
# Exclude the redundant repository checkout and its .git directory.
zip_path=RUN.with_suffix('.zip')
import zipfile
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in RUN.rglob('*'):
        if p.is_file() and REPO not in p.parents:
            z.write(p,str(p.relative_to(RUN)))
print(zip_path)
files.download(str(zip_path))
